# Project B — WestQuant Hamiltonian Representation Explorer (QoolQit)

**QoolQit version: 1.4.0**

Search over mathematically valid Hamiltonian representations of one
underlying optimization problem. Each representation is verified and
classified into exactly one of five equivalence classes:

- `EXACT_EQUIVALENT`: `E'(T(x)) = a E(x) + b` for all states
- `GROUND_STATE_EQUIVALENT`: optimum set preserved
- `SAME_PROBLEM_DIFFERENT_DYNAMICS`: same problem, different dynamics
- `APPROXIMATE`: approximate relationship
- `INVALID`: unjustified

In [ ]:
import numpy as np, qoolqit
print('QoolQit version:', qoolqit.__version__)
from westquant_qoolqit.common import BinaryQuadraticHamiltonian, solve_exact, QOOLQIT_VERSION
from westquant_qoolqit.hamiltonian_explorer import (
    HamiltonianRepresentationExplorer, positive_scale, bit_complement,
    mwis_penalty, classify_realizability,
)
from westquant_qoolqit.benchmarks import mwis_path
assert QOOLQIT_VERSION == qoolqit.__version__

## 1. Problem: MWIS

In [ ]:
h, info = mwis_path(n=5, seed=42)
exact = solve_exact(h)
print('Optimum:', exact.optimum_states.tolist(), 'E=', exact.optimum_energy)

## 2. Exact transforms

Verify that scaling, permutation, and bit-complement are truly
`EXACT_EQUIVALENT` by exhaustive state-by-state comparison.

In [ ]:
tr_scale = positive_scale(h, a=2.0, b=1.0)
tr_complement = bit_complement(h, S=[0, 2, 4])
from westquant_qoolqit.common import verify_equivalence
for name, tr in [('scale', tr_scale), ('complement', tr_complement)]:
    rep = verify_equivalence(h, tr.hamiltonian, forward_map=tr.forward_map)
    print(f'{name}: {rep.classification.value} max_err={rep.max_energy_error:.2e}')

## 3. MWIS penalty family

Different penalty strengths U yield `GROUND_STATE_EQUIVALENT` Hamiltonians
with the same optimum but different landscapes (gaps, degeneracies).

In [ ]:
explorer = HamiltonianRepresentationExplorer(
    transforms=['mwis_penalty'], verify=True, seed=42,
)
res = explorer.explore(h, mwis_info=info)
for c in res.candidates:
    if c.transform_name == 'mwis_penalty':
        lm = c.landscape
        print(f'  U={c.transform_parameters["U"]:.2f} eq={c.equivalence_class.value} '
              f'gap={lm.classical_gap:.3f} deg={lm.ground_state_degeneracy}')

## 4. Physical realizability

Native Rydberg interactions are repulsive (J > 0). Bit-complement can flip
signs, making a Hamiltonian non-native. The classifier flags this before
expensive embedding.

In [ ]:
for c in res.candidates:
    rz = c.realizability
    print(f'  {c.id}: native={rz.native_pair_interactions} '
          f'n_neg={rz.n_negative_interactions} range={rz.dynamic_range:.2f}')

## 5. Conclusion

The same MWIS problem admits multiple valid Hamiltonian representations
with different landscapes and physical realizability. Treating the
Hamiltonian as an optimization variable (not a fixed choice) matters.